# 🐍 Resumo de POO para a Prova
### Programação Orientada a Objetos em Python — teoria + código executável

**Como usar este notebook:** leia a célula de teoria e rode a célula de código logo abaixo
(`Shift+Enter`). Os resultados já estão salvos, então dá para estudar só lendo — mas rodar
ajuda a fixar.

---

### Sumário

| # | Tópico | Ideia central em 1 linha |
|---|--------|--------------------------|
| 1 | Introdução à POO e Abstração | Classe = molde; objeto = instância materializada |
| 2 | Construtores (`__new__` / `__init__`) | `__new__` aloca, `__init__` inicializa |
| 3 | Encapsulamento e Modificadores | `public`, `_protected`, `__private` + `@property` |
| 4 | Atributos de classe, `@classmethod`, `@staticmethod` | Estado compartilhado x utilitários |
| 5 | Relacionamentos | Agregação (losango aberto) x Composição (losango fechado) |
| 6 | Herança e Problema do Diamante | Herança múltipla resolvida pelo MRO |
| 7 | Polimorfismo e Injeção de Dependência | Mesma chamada, comportamentos diferentes; menos acoplamento |
| 8 | Classes e Métodos Abstratos | Contrato que não pode ser instanciado (`abc`) |
| 9 | Cola rápida / pegadinhas | Dunders, armadilhas clássicas de prova |
| 10 | Simulado com gabarito | 10 questões para se testar |

---
## 1. Introdução à POO e Abstração

**Abstração** = isolar as características essenciais de um elemento do mundo real para levar
para o código (ignorar o que não importa para o problema).

- **Classe** → estrutura que serve de *molde/especificação*. Define:
  - **Atributos**: os estados (dados) do objeto.
  - **Métodos**: os comportamentos (ações) do objeto.
- **Objeto** → a *instância* da classe, criada em **tempo de execução**.
  Todos os objetos de uma mesma classe têm a **mesma estrutura** (mesmos atributos e métodos),
  mas cada um guarda os **seus próprios valores**.

No UML, a classe é desenhada como um retângulo dividido em 3 partes: **nome | atributos | métodos**.

In [3]:
class Pessoa:
    def __init__(self, nome, idade):
        self.nome = nome        # atributo de instância
        self.idade = idade

    def apresentar(self):       # método de instância
        return f"Oi, sou {self.nome} e tenho {self.idade} anos."


p1 = Pessoa("Ana", 22)
p2 = Pessoa("Bruno", 30)

print(p1.apresentar())
print(p2.apresentar())

print("---")
print("p1 é instância de Pessoa?", isinstance(p1, Pessoa))
print("p1 e p2 são o MESMO objeto?", p1 is p2, "(mesma estrutura, valores diferentes)")
print("id(p1) =", id(p1), "| id(p2) =", id(p2))
print("estado de p1:", p1.__dict__)
print("estado de p2:", p2.__dict__)

Oi, sou Ana e tenho 22 anos.
Oi, sou Bruno e tenho 30 anos.
---
p1 é instância de Pessoa? True
p1 e p2 são o MESMO objeto? False (mesma estrutura, valores diferentes)
id(p1) = 2146907212160 | id(p2) = 2146908151056
estado de p1: {'nome': 'Ana', 'idade': 22}
estado de p2: {'nome': 'Bruno', 'idade': 30}


---
## 2. Construtores: `__new__` e `__init__`

O ciclo de vida de um objeto em Python tem duas etapas:

| Etapa | Método | Papel |
|-------|--------|-------|
| 1 | `__new__(cls, ...)` | **Aloca o espaço em memória** e devolve a nova instância. Recebe a **classe** (`cls`). |
| 2 | `__init__(self, ...)` | **Inicializa os atributos** da instância já criada. Recebe o **objeto** (`self`). |

- Quem chama o `__new__` é o próprio Python; ele quem *evoca* o `__init__`.
- Na prática você quase sempre só escreve o `__init__`. O `__new__` só é necessário em casos
  especiais (tipos imutáveis, *singleton*, cache de objetos...).
- O `__init__` pode receber parâmetros **com ou sem valor padrão**.
- Atenção: `__init__` **não é o criador** do objeto — ele só configura algo que já existe.

In [30]:
class Exemplo:
    def __new__(cls, *args, **kwargs):
        _ = args, kwargs                 # aceita argumentos para repassá-los ao __init__
        print("1) __new__  -> alocando memória")
        instancia = super().__new__(cls)     # delega para o __new__ de object
        return instancia

    def __init__(self, valor):
        print("2) __init__ -> inicializando atributos")
        self.valor = valor


obj = Exemplo(10)
print("objeto pronto:", obj.valor)

1) __new__  -> alocando memória
2) __init__ -> inicializando atributos
objeto pronto: 10


**`__init__` com valores padrão** (parametrizável, mas opcional na criação):

In [5]:
class Aluno:
    def __init__(self, nome, curso="TADS", nota=0.0):
        self.nome = nome
        self.curso = curso
        self.nota = nota


a1 = Aluno("Fulano")                              # usa todos os padrões
a2 = Aluno("Beltrano", "TSI")        # troca só o curso
a3 = Aluno("Siclano", nota=9.5)                  # usa argumento nomeado

for a in (a1, a2, a3):
    print(f"{a.nome:<6} | {a.curso:<18} | {a.nota}")

Fulano | TADS               | 0.0
Beltrano | TSI                | 0.0
Siclano | TADS               | 9.5


---
## 3. Encapsulamento e Modificadores de Acesso

Encapsulamento = esconder o estado interno e expor só o necessário, protegendo o objeto de
alterações indevidas (mantendo a lógica de validação **oculta** para quem usa a classe).

| Modificador | Sintaxe | Acesso |
|-------------|---------|--------|
| Público | `self.nome` | livre, de qualquer lugar |
| Protegido | `self._nome` | própria classe e **subclasses** (convenção) |
| Privado | `self.__nome` | somente dentro da própria classe |

⚠️ Em Python nada disso é imposto pelo compilador: `_` e `__` são **convenções** e o `__`
sofre **name mangling** (o atributo vira `_Classe__atributo`). A ideia é: "não encoste, isso é
detalhe interno".

In [26]:
class Conta:
    def __init__(self, titular, saldo=0.0):
        self.titular = titular      # público
        self._agencia = "0001"      # protegido (convenção)
        self.__saldo = saldo        # privado -> vira _Conta__saldo

    def depositar(self, valor):
        if valor > 0:
            self.__saldo += valor

    def ver_saldo(self):            # acesso controlado pelo método
        return self.__saldo


c = Conta("Ana", 100)
print("público :", c.titular)
print("protegido:", c._agencia)
c.depositar(50)
print("saldo via método:", c.ver_saldo())

print("--- tentando acessar o privado de fora ---")
try:
    print(c.__saldo)
except AttributeError as e:
    print("AttributeError:", e)

print("--- name mangling (a 'quebra' possível) ---")
print("c._Conta__saldo =", getattr(c, "_Conta__saldo"), "-> é proteção contra acidente, não segurança")
print("atributos do objeto:", sorted(vars(c)))

público : Ana
protegido: 0001
saldo via método: 150
--- tentando acessar o privado de fora ---
AttributeError: 'Conta' object has no attribute '__saldo'
--- name mangling (a 'quebra' possível) ---
c._Conta__saldo = 150 -> é proteção contra acidente, não segurança
atributos do objeto: ['_Conta__saldo', '_agencia', 'titular']


### Getters e Setters com `@property`

O decorator `@property` transforma um **método** em um **atributo de acesso controlado**:
quem usa a classe escreve `conta.saldo` (sem parênteses), mas por trás roda o getter —
e a validação continua escondida.

- `@property` → define o **getter**.
- `@nome.setter` → define o **setter** (validação entra aqui).
- Só com getter → atributo **somente leitura**.

In [27]:
class ContaSegura:
    def __init__(self, titular, saldo=0.0):
        self.titular = titular
        self.saldo = saldo          # passa pelo setter (validação já na criação)

    @property
    def saldo(self):                # GETTER
        return self._saldo

    @saldo.setter
    def saldo(self, valor):         # SETTER com validação
        if valor < 0:
            raise ValueError("Saldo não pode ser negativo!")
        self._saldo = valor

    @property
    def resumo(self):               # property SOMENTE LEITURA (não tem setter)
        return f"{self.titular}: R$ {self._saldo:.2f}"


conta = ContaSegura("Ana", 100)
print(conta.resumo)

conta.saldo = 250
print("depois do setter:", conta.resumo)

try:
    conta.saldo = -50
except ValueError as e:
    print("ValueError:", e)

try:
    setattr(conta, "resumo", "hack")
except AttributeError as e:
    print("AttributeError:", e)

Ana: R$ 100.00
depois do setter: Ana: R$ 250.00
ValueError: Saldo não pode ser negativo!
AttributeError: property 'resumo' of 'ContaSegura' object has no setter


---
## 4. Atributos de Classe, `@classmethod` e `@staticmethod`

- **Atributo de classe**: declarado no *corpo da classe*, **fora** de qualquer método.
  É compartilhado por **todas** as instâncias (uma cópia só, acessível por `Classe.atributo`).

- 🕳️ **Armadilha do mutável compartilhado**: se o atributo de classe for uma coleção mutável
  (lista, dicionário, set), ela é criada **uma única vez** e o estado **vaza entre os objetos**.
  Coleções mutáveis devem nascer **dentro do `__init__`** (aí cada instância tem a sua).

### Comparando os decorators

| Decorator | 1º parâmetro | Serve para | Chamada |
|-----------|--------------|-----------|---------|
| *(nenhum)* — método de instância | `self` | operar sobre **um objeto** | `obj.metodo()` |
| `@classmethod` | `cls` (a classe) | **construtores alternativos** / alterar dados da classe | `Classe.metodo()` |
| `@staticmethod` | nada | **função utilitária** agrupada por tema | `Classe.metodo()` |

In [8]:
class Contador:
    total = 0                       # atributo de CLASSE (compartilhado)

    def __init__(self, nome):
        self.nome = nome
        Contador.total += 1         # incrementa o contador global da classe


print(Contador.total)
a, b, c = Contador("A"), Contador("B"), Contador("C")
print("objetos criados:", Contador.total, "| visto pela instância:", a.total)

0
objetos criados: 3 | visto pela instância: 3


### 🕳️ Demonstração da armadilha do mutável compartilhado

In [9]:
class TurmaErrada:
    alunos = []             # ❌ criada UMA vez no corpo da classe: compartilhada!


class TurmaCerta:
    def __init__(self):
        self.alunos = []    # ✅ nasce dentro do __init__: uma por objeto


e1, e2 = TurmaErrada(), TurmaErrada()
e1.alunos.append("Ana")
print("ERRADO -> e1.alunos =", e1.alunos)
print("ERRADO -> e2.alunos =", e2.alunos, " <-- vazou estado entre objetos!")
print("é a mesma lista?", e1.alunos is e2.alunos)

c1, c2 = TurmaCerta(), TurmaCerta()
c1.alunos.append("Ana")
print("\nCERTO  -> c1.alunos =", c1.alunos)
print("CERTO  -> c2.alunos =", c2.alunos, " <-- independentes, como deve ser")

ERRADO -> e1.alunos = ['Ana']
ERRADO -> e2.alunos = ['Ana']  <-- vazou estado entre objetos!
é a mesma lista? True

CERTO  -> c1.alunos = ['Ana']
CERTO  -> c2.alunos = []  <-- independentes, como deve ser


### `@classmethod` — construtor alternativo (método de fábrica)

In [22]:
from datetime import date


class Data:
    formato = "%d/%m/%Y"

    def __init__(self, dia, mes, ano):
        self.dia, self.mes, self.ano = dia, mes, ano

    @classmethod
    def de_iso(cls, texto):                # construtor ALTERNATIVO
        ano, mes, dia = (int(x) for x in texto.split("-"))
        return cls(dia, mes, ano)          # cls = a própria classe

    @classmethod
    def alterar_formato(cls, novo):        # altera dado da CLASSE
        cls.formato = novo

    def __str__(self):
        return date(self.ano, self.mes, self.dia).strftime(self.formato)


d1 = Data(30, 9, 2026)
d2 = Data.de_iso("2026-09-30")             # chamado na CLASSE, sem instância pronta
print(d1, "|", d2, "| iguais?", str(d1) == str(d2))

Data.alterar_formato("%Y-%m-%d")
print("depois de alterar o formato:", d1)
print("formato da classe agora:", Data.formato)

30/09/2026 | 30/09/2026 | iguais? True
depois de alterar o formato: 2026-09-30
formato da classe agora: %Y-%m-%d


### `@staticmethod` — função utilitária dentro da classe

Não recebe `self` nem `cls`. Está ali apenas por **organização temática** (poderia ser uma função
solta no módulo).

In [11]:
class Validador:
    @staticmethod
    def cpf_formato_ok(cpf):               # não usa self nem cls
        return len([d for d in cpf if d.isdigit()]) == 11

    @staticmethod
    def media(notas):
        return sum(notas) / len(notas) if notas else 0


print(Validador.cpf_formato_ok("123.456.789-01"))
print(Validador.cpf_formato_ok("123.456"))
print(round(Validador.media([7, 8, 10]), 2))

True
False
8.33


---
## 5. Relacionamentos entre Classes

| Aspecto | **Agregação** | **Composição** |
|---------|---------------|----------------|
| Relação | "todo/parte" fraca = **posse** | "todo/parte" forte = **criação** |
| Ciclo de vida | a parte **existe independente** do todo | a parte **não sobrevive** sem o todo |
| Quem cria a parte | vem **de fora** (passada ao todo) | o **todo cria** a parte internamente |
| UML | **losango ABERTO** (vazado) na ponta do "todo" | **losango FECHADO** (preenchido) |
| Exemplo | Departamento ↔ Professor | Pedido ↔ ItemPedido / Carro ↔ Motor |

💡 Dica de prova: pergunte *"se eu destruir o todo, a parte continua existindo?"* → **sim = agregação**, **não = composição**.

In [12]:
# ---------- AGREGAÇÃO: losango aberto ----------
class Professor:
    def __init__(self, nome):
        self.nome = nome


class Departamento:
    def __init__(self, nome):
        self.nome = nome
        self.professores = []                  # recebe objetos prontos de fora

    def contratar(self, professor):
        self.professores.append(professor)     # POSSE, não criação


prof = Professor("Ana")                        # o professor existe por conta própria
dep = Departamento("Computação")
dep.contratar(prof)

print("Professores do depto:", [p.nome for p in dep.professores])
del dep                                        # destruindo o "todo"
print("Professor continua existindo? ->", prof.nome)

Professores do depto: ['Ana']
Professor continua existindo? -> Ana


In [23]:
# ---------- COMPOSIÇÃO: losango fechado ----------
class ItemPedido:
    def __init__(self, descricao, preco):
        self.descricao = descricao
        self.preco = preco


class Pedido:
    def __init__(self, numero):
        self.numero = numero
        self.itens = []

    def adicionar(self, descricao, preco):
        item = ItemPedido(descricao, preco)    # o TODO CRIA A PARTE (por dentro!)
        self.itens.append(item)

    @property
    def total(self):
        return sum(i.preco for i in self.itens)


pedido = Pedido(101)
pedido.adicionar("Teclado", 150.0)
pedido.adicionar("Mouse", 80.0)

print("Itens:", [i.descricao for i in pedido.itens])
print("Total: R$", pedido.total)
print("Neste modelo, o Pedido cria os ItemPedido internamente -> composição")

Itens: ['Teclado', 'Mouse']
Total: R$ 230.0
Neste modelo, o Pedido cria os ItemPedido internamente -> composição


> **Observação de prova:** em Python existe *garbage collector*, então a parte não "morre" com um
> `del` explícito se alguém ainda referencia-la. A diferença agregação x composição está no
> **modelo conceitual** (UML): **quem cria** a parte e **se ela depende** do todo — não no operador
> `del`.

---
## 6. Herança e o Problema do Diamante

- **Herança**: a subclasse **herda** atributos e métodos da(s) classe(s) base ("é um").
- **`super()`**: acessa a implementação da classe **pai** (não repita código herdado).
- **Herança múltipla**: a subclasse herda de **mais de uma** classe base ao mesmo tempo.

### Problema do Diamante

Quando duas superclasses possuem **métodos com o mesmo nome** (ex.: `A` no topo, `B` e `C` herdando
de `A`, e `D` herdando de `B` e `C` — o desenho forma um losango/diamante), a dúvida é:
**qual método executa?**

O Python resolve com o **MRO (Method Resolution Order)**, calculado pela *linearização C3*:
a prioridade vai da **esquerda para a direita** na declaração `class D(B, C)`.
Você inspeciona com `Classe.__mro__` (ou `Classe.mro()`).

In [14]:
class Animal:
    def __init__(self, nome):
        self.nome = nome

    def falar(self):
        return "..."


class Cachorro(Animal):              # herança simples
    def falar(self):                 # SOBRESCRITA (overriding)
        return "Woof!"


class Gato(Animal):
    def falar(self):
        return "Miau!"


for bicho in (Cachorro("Rex"), Gato("Tom")):
    print(f"{bicho.nome}: {bicho.falar()}")   # mesmo método, comportamentos diferentes

Rex: Woof!
Tom: Miau!


### O diamante na prática

In [15]:
class A:
    def quem(self):
        return "A"


class B(A):
    def quem(self):
        return "B -> " + super().quem()      # super() segue o MRO!


class C(A):
    def quem(self):
        return "C -> " + super().quem()


class D(B, C):                               # herança múltipla
    def quem(self):
        return "D -> " + super().quem()


print("MRO de D:", [c.__name__ for c in D.__mro__])
print("Chamada de D().quem():", D().quem())
print()
print("Leitura: procura por D, depois B (esquerda), depois C (direita), depois A, depois object.")
print("O super() de B NÃO vai direto para A: ele vai para o próximo da fila -> C.")

MRO de D: ['D', 'B', 'C', 'A', 'object']
Chamada de D().quem(): D -> B -> C -> A

Leitura: procura por D, depois B (esquerda), depois C (direita), depois A, depois object.
O super() de B NÃO vai direto para A: ele vai para o próximo da fila -> C.


---
## 7. Polimorfismo e Injeção de Dependência

**Polimorfismo** = objetos de classes **diferentes** respondem à **mesma chamada de método**,
cada um com o seu comportamento. Em Python isso é natural (*duck typing*): se tem `area()`, serve —
não importa de quem herdou.

- **Sobrescrita de método (*overriding*)**: a subclasse reescreve a implementação de um método herdado.
  (Diferente de *overloading*, que em Python não existe do jeito de Java — usar valor padrão ou `*args`.)

**Injeção de Dependência (DI)**: em vez de a classe **criar** as suas dependências por dentro, ela
**recebe** objetos prontos (tipicamente pelo `__init__`). Resultado: **menos acoplamento** e
facilidade de trocar/testar peças.

In [16]:
class Circulo:
    def __init__(self, raio):
        self.raio = raio

    def area(self):
        return 3.14159 * self.raio ** 2


class Retangulo:
    def __init__(self, base, altura):
        self.base, self.altura = base, altura

    def area(self):
        return self.base * self.altura


class Quadrado:
    def __init__(self, lado):
        self.lado = lado

    def area(self):
        return self.lado ** 2


formas = [Circulo(2), Retangulo(3, 4), Quadrado(5)]
for f in formas:
    print(f"{type(f).__name__:<10} -> área = {f.area():.2f}")

print("\nUma única função serve para todas (POLIMORFISMO):")
def relatorio(objetos):
    return ", ".join(f"{type(o).__name__}: {o.area():.1f}" for o in objetos)

print(relatorio(formas))

Circulo    -> área = 12.57
Retangulo  -> área = 12.00
Quadrado   -> área = 25.00

Uma única função serve para todas (POLIMORFISMO):
Circulo: 12.6, Retangulo: 12.0, Quadrado: 25.0


### Injeção de Dependência: acoplado x desacoplado

In [17]:
class EmailService:
    def enviar(self, destino, msg):
        print(f"[E-MAIL] para {destino}: {msg}")


class SmsService:
    def enviar(self, destino, msg):
        print(f"[SMS]    para {destino}: {msg}")


# ❌ SEM injeção: a classe decide tudo, impossível trocar sem editar o código
class PedidoAcoplado:
    def finalizar(self, cliente):
        EmailService().enviar(cliente, "Seu pedido foi confirmado")


# ✅ COM injeção: a dependência chega de fora pelo __init__ (injeção por construtor)
class PedidoService:
    def __init__(self, notificador):
        self.notificador = notificador          # depende de uma ABSTRAÇÃO (basta ter .enviar)

    def finalizar(self, cliente):
        print(f"Pedido de {cliente} processado.")
        self.notificador.enviar(cliente, "Seu pedido foi confirmado")


PedidoAcoplado().finalizar("Ana")
print()
PedidoService(EmailService()).finalizar("Bruno")     # troca o canal...
print()
PedidoService(SmsService()).finalizar("Carla")       # ...sem tocar em PedidoService


class NotificadorFake:                                # ótimo para testar
    def __init__(self):
        self.enviados = []

    def enviar(self, destino, msg):
        self.enviados.append((destino, msg))


fake = NotificadorFake()
PedidoService(fake).finalizar("Diego")
print("\nTeste automatizado -> mensagens registradas:", fake.enviados)

[E-MAIL] para Ana: Seu pedido foi confirmado

Pedido de Bruno processado.
[E-MAIL] para Bruno: Seu pedido foi confirmado

Pedido de Carla processado.
[SMS]    para Carla: Seu pedido foi confirmado
Pedido de Diego processado.

Teste automatizado -> mensagens registradas: [('Diego', 'Seu pedido foi confirmado')]


---
## 8. Classes e Métodos Abstratos

- **Classe abstrata** define um **contrato**: um conjunto de métodos que as subclasses
  **obrigatoriamente** devem implementar. Ela é um "molde incompleto".
- ❌ **Não pode ser instanciada diretamente** — só serve de base para herança.
- Em Python usa-se o módulo **`abc`**:

```python
from abc import ABC, abstractmethod

class Forma(ABC):
    @abstractmethod
    def area(self): ...
```

- Se uma subclasse **não implementar** todos os métodos abstratos, ela também não pode ser
  instanciada (o Python avisa na hora da criação).

### Notação UML

- Nome da **classe abstrata** e dos **métodos abstratos** em ***itálico***.
- A **herança de uma classe abstrata** usa uma linha **contínua** com seta de ponta aberta.
  Linha tracejada com seta de ponta aberta é usada para realização de interface/contrato.

In [28]:
from abc import ABC, abstractmethod


class Forma(ABC):                        # classe abstrata
    def __init__(self, nome):
        self.nome = nome

    @abstractmethod
    def area(self):                      # CONTRATO: quem herdar deve implementar
        ...

    @abstractmethod
    def perimetro(self):
        ...

    def descricao(self):                 # método concreto: já pronto para usar
        return f"{self.nome}: área={self.area():.2f}, perímetro={self.perimetro():.2f}"


print("Tentando instanciar a classe abstrata:")
try:
    f = Forma("genérica")  # type: ignore[abstract]
except TypeError as e:
    print("TypeError:", e)

Tentando instanciar a classe abstrata:
TypeError: Can't instantiate abstract class Forma without an implementation for abstract methods 'area', 'perimetro'


In [29]:
class QuadradoForma(Forma):
    def __init__(self, lado):
        super().__init__("Quadrado")
        self.lado = lado

    def area(self):
        return self.lado ** 2

    def perimetro(self):
        return 4 * self.lado


class CirculoForma(Forma):
    def __init__(self, raio):
        super().__init__("Círculo")
        self.raio = raio

    def area(self):
        return 3.14159 * self.raio ** 2

    def perimetro(self):
        return 2 * 3.14159 * self.raio


print(QuadradoForma(3).descricao())
print(CirculoForma(1).descricao())
print("QuadradoForma é uma Forma?", isinstance(QuadradoForma(3), Forma))

print("\nSubclasse que NÃO cumpre o contrato:")
class Incompleta(Forma):
    def area(self):
        return 0
    # faltou perimetro()

try:
    Incompleta("incompleta")  # type: ignore[abstract]
except TypeError as e:
    print("TypeError:", e)

Quadrado: área=9.00, perímetro=12.00
Círculo: área=3.14, perímetro=6.28
QuadradoForma é uma Forma? True

Subclasse que NÃO cumpre o contrato:
TypeError: Can't instantiate abstract class Incompleta without an implementation for abstract method 'perimetro'


---
## 9. 🧠 Cola rápida e pegadinhas clássicas de prova

### Dunders que costumam cair

| Dunder | Quando roda | Observação |
|--------|-------------|------------|
| `__new__(cls)` | antes de tudo, na criação | aloca memória, retorna a instância |
| `__init__(self)` | na criação | inicializa atributos ("construtor") |
| `__str__` | `print(obj)`, `str(obj)` | versão "bonita", para leitura |
| `__repr__` | `repr(obj)`, no interpretador | versão técnica / debug |
| `__mro__` | consulta | ordem de resolução de métodos |
| `__dict__` | consulta | atributos de instância |
| `__eq__`, `__len__` | `==`, `len()` | comportamentos especiais |

### As 7 pegadinhas que mais derrubam

1. `__new__` **aloca**, `__init__` **inicializa** — e `__init__` não cria o objeto.
2. `__privado` é **name mangling** (`_Classe__privado`), não segurança real.
3. Coleção mutável no **corpo da classe** → estado **compartilhado** entre objetos. Use `__init__`.
4. **Argumento padrão mutável** (`def f(x, lista=[])`) → a mesma lista persiste entre chamadas.
5. Herança múltipla: quem manda é o **MRO** (`Classe.__mro__`), esquerda → direita.
6. `super()` não é "o pai da minha classe", é **o próximo da fila do MRO**.
7. Classe abstrata **não instancia**; o erro é `TypeError: Can't instantiate abstract class`.

In [20]:
# PEGADINHA 4 na prática: argumento padrão mutável
def adicionar_item_errado(item, carrinho=[]):     # ❌ o default é criado UMA vez
    carrinho.append(item)
    return carrinho


print(adicionar_item_errado("arroz"))
print(adicionar_item_errado("feijão"))            # 'arroz' ainda está lá!
print(adicionar_item_errado("macarrão"))


def adicionar_item_certo(item, carrinho=None):    # ✅ cria uma lista nova a cada chamada
    if carrinho is None:
        carrinho = []
    carrinho.append(item)
    return carrinho


print()
print(adicionar_item_certo("arroz"))
print(adicionar_item_certo("feijão"))

['arroz']
['arroz', 'feijão']
['arroz', 'feijão', 'macarrão']

['arroz']
['feijão']


In [21]:
# PEGADINHA 6 na prática: super() segue o MRO completo
class A:
    def registrar(self):
        return ["A"]


class B(A):
    def registrar(self):
        return ["B"] + super().registrar()


class C(A):
    def registrar(self):
        return ["C"] + super().registrar()


class D(B, C):
    def registrar(self):
        return ["D"] + super().registrar()


print("MRO:", [c.__name__ for c in D.__mro__])
print("Cadeia executada:", D().registrar())
print("Repare: A aparece UMA vez, e C entra antes de A mesmo B não mencionando C.")

MRO: ['D', 'B', 'C', 'A', 'object']
Cadeia executada: ['D', 'B', 'C', 'A']
Repare: A aparece UMA vez, e C entra antes de A mesmo B não mencionando C.


---
## 10. 📝 Simulado rápido

Responda mentalmente (ou escreva num papel) antes de olhar o gabarito:

1. Qual a diferença entre **classe** e **objeto**?
2. Quem **aloca a memória**: `__new__` ou `__init__`?
3. O que significa `self.__saldo` na prática, depois da compilação do nome?
4. Para que serve o `@property` em relação ao encapsulamento?
5. Por que declarar `itens = []` no corpo da classe é um erro de projeto? Onde deve ficar?
6. Diferencie `@classmethod` de `@staticmethod` (primeiro parâmetro e uso típico).
7. **Agregação** x **composição**: qual usa losango fechado no UML? Cite um exemplo de cada.
8. Em `class D(B, C)`, se `B` e `C` têm um método com o mesmo nome, qual executa? Como o Python decide?
9. O que é **polimorfismo**? E o que é **injeção de dependência** (cite o benefício)?
10. Por que uma classe que herda de `ABC` e não implementa um `@abstractmethod` não pode ser instanciada?

<details>
<summary><b>👀 GABARITO — clique para abrir</b></summary>

1. **Classe** é o molde/especificação (atributos + métodos); **objeto** é a instância criada em
   tempo de execução, com valores próprios.
2. **`__new__`** aloca a memória e devolve a instância; o `__init__` apenas inicializa os atributos.
3. Vira `_Classe__saldo` (**name mangling**): esconde o nome, mas não impede o acesso — é convenção.
4. Cria getters/setters que parecem atributos comuns (`obj.saldo`), permitindo **validar**
   e esconder a lógica de acesso, mantendo o encapsulamento.
5. Porque a lista é criada **uma só vez** e fica **compartilhada** por todas as instâncias (vaza
   estado). Deve ser criada **dentro do `__init__`**, em `self.itens = []`.
6. `@classmethod` recebe **`cls`** e serve para **construtores alternativos** / mexer em dados da
   classe; `@staticmethod` não recebe `self` nem `cls`, é **função utilitária** agrupada por tema.
7. **Composição** usa losango **fechado** (parte criada pelo todo, não sobrevive sem ele — ex.:
   Pedido/ItemPedido). Agregação usa losango **aberto** (partes independentes — ex.: Departamento/Professor).
8. Executa o da **classe mais à esquerda** na ordem do **MRO** (`D → B → C → A → object`), definido
   pela linearização C3 — verificável com `D.__mro__`.
9. **Polimorfismo**: classes diferentes respondendo à mesma chamada de método com comportamentos
   próprios. **Injeção de dependência**: receber as dependências de fora (via `__init__`) em vez de
   criá-las internamente → **menos acoplamento** e mais facilidade de testar/trocar implementações.
10. Porque ela **não cumpre o contrato**: continua abstrata, e o Python lança
    `TypeError: Can't instantiate abstract class ... with abstract method ...`.

</details>

---

### ✅ Checklist de véspera

- [ ] Sei escrever uma classe com `__init__`, atributos e métodos, e criar objetos.
- [ ] Sei explicar `public` / `_protegido` / `__privado` e fazer getter/setter com `@property`.
- [ ] Sei demonstrar a armadilha do mutável compartilhado (e como corrigir).
- [ ] Sei escrever um `@classmethod` de fábrica e um `@staticmethod` utilitário.
- [ ] Sei diferenciar agregação de composição e desenhar os losangos no UML.
- [ ] Sei explicar/consultar o MRO em herança múltipla.
- [ ] Sei implementar uma hierarquia com `ABC` + `@abstractmethod`.
- [ ] Sei escrever um exemplo de polimorfismo e um de injeção de dependência.

**Boa prova! 🚀**